In [0]:
from utils.logger import get_logger
from etl_config.gold_dim_config import DIM_CONFIG, GOLD
from etl_config.gold_fact_config import FACT_CONFIG

logger = get_logger("gold_sanity_check")

### SK unique / SCD2 Integrity

In [0]:

for table_name, cfg in DIM_CONFIG.items():
    logger.info(f"========================")
    logger.info(f"Validating {table_name}")
    logger.info(f"------------------------")
    if cfg.surrogate_key_name:
        sk_unique_cnt = spark.sql(f"""
            select count(*) as cnt
            from (
                select {cfg.surrogate_key_name}, count(*) as n
                from {cfg.target_table}
                group by {cfg.surrogate_key_name}
                having n > 1
            )
        """).collect()[0].cnt

        if sk_unique_cnt > 0:
            logger.warning(f"ERROR: {table_name} has {sk_unique_cnt} records with duplicate surrogate keys")
        else:
            logger.info(f"OK: {table_name} has no records with duplicate surrogate keys")


    if cfg.tracked_columns:
        nk = cfg.natural_key

        multi_current = spark.sql(f"""
            select count(*) as cnt
            from (
                select {nk}, count(*) as n
                from {cfg.target_table}
                where is_current = true
                group by {nk}
                having n > 1
            )
        """).collect()[0].cnt

        if multi_current > 0:
            logger.warning(f"ERROR: {table_name} has {multi_current} records with multiple current versions")
        else:
            logger.info(f"OK: {table_name} has no records with multiple current versions")

        expired_no_valid_to = spark.sql(f"""
            select count(*) as cnt
            from {cfg.target_table}
            where is_current = false and valid_to is null
        """).collect()[0].cnt

        if expired_no_valid_to > 0:
            logger.warning(f"ERROR: {table_name} has {expired_no_valid_to} expired records with no valid_to date")
        else:
            logger.info(f"OK: {table_name} has no expired records with no valid_to date")

        current_with_valid_to = spark.sql(f"""
            select count(*) as cnt
            from {cfg.target_table}
            where is_current = true and valid_to is not null
        """).collect()[0].cnt

        if current_with_valid_to > 0:
            logger.warning(f"ERROR: {table_name} has {current_with_valid_to} current records with valid_to date")
        else:
            logger.info(f"OK: {table_name} has no current records with valid_to date")

        invalid_dates = spark.sql(f"""
            select count(*) as cnt
            from {cfg.target_table}
            where is_current = FALSE and valid_from >= valid_to
        """).collect()[0].cnt

        if invalid_dates > 0:
            logger.warning(f"ERROR: {table_name} has {invalid_dates} records with invalid dates")
        else:
            logger.info(f"OK: {table_name} has no records with invalid dates")


### DIM Date

In [0]:
date_gaps = spark.sql(f"""
    select count(*) as gap_count
    from (
        select datediff(
                lead(full_date) over (order by full_date),
                full_date 
            ) as days_diff
        from {GOLD}.dim_date
    )
    where days_diff > 1
""").collect()[0].gap_count

if date_gaps > 0:
    logger.warning(f"ERROR: {date_gaps} gaps in date dimension")
else:
    logger.info(f"OK: {date_gaps} gaps in date dimension")

### Fact sanity checks

In [0]:
%sql
select count(*) as orphan_count 
from acme_catalog.gold.fact_orders f
left join acme_catalog.gold.dim_shippers on f.shipper_sk = dim_shippers.shipper_sk
where dim_shippers.shipper_sk is null and f.shipper_sk != -1;

In [0]:
%sql
-- Grain unique
select order_id, line_no, count(*) as cnt
from acme_catalog.gold.fact_order_lines
group by order_id, line_no
having count(*) > 1

In [0]:
%sql
select order_id, count(*) as cnt
from acme_catalog.gold.fact_orders
group by order_id
having cnt > 1

In [0]:
%sql
select order_id, line_no, count(*) as cnt
from acme_catalog.gold.fact_shipments
group by order_id, line_no
having cnt > 1


In [0]:
%sql
with cnt_fact_orders as (
    select count(*) cnt 
    from acme_catalog.gold.fact_orders
),
cnt_orders as (
    select count(*) cnt 
    from acme_catalog.silver.orders
)
select f.cnt as fact_count, 
       s.cnt as silver_count, 
       f.cnt-s.cnt as row_diff
from cnt_fact_orders f
cross join cnt_orders s